# Bronze Layer
Load raw CSVs from Volume into bronze schema tables.  
All columns loaded as STRING — Silver layer handles type casting.

## Setup Connection

In [1]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "bronze",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
VOL_SCHEMA = os.environ.get("CLICKZETTA_SCHEMA", "public")  # schema where Volume lives
VOLUME     = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Define Ingestion Configuration

In [2]:
INGESTION_CONFIG = [
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_crm/cust_info.csv",     "table": "bronze.crm_cust_info"},
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_crm/prd_info.csv",      "table": "bronze.crm_prd_info"},
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_crm/sales_details.csv", "table": "bronze.crm_sales_details"},
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_erp/CUST_AZ12.csv",     "table": "bronze.erp_cust_az12"},
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_erp/LOC_A101.csv",      "table": "bronze.erp_loc_a101"},
    {"path": f"vol://{VOL_SCHEMA}.{VOLUME}/source_erp/PX_CAT_G1V2.csv",  "table": "bronze.erp_px_cat_g1v2"},
]

## Ingest Files into Bronze Tables

In [3]:
for item in INGESTION_CONFIG:
    print(f"Ingesting → {item['table']}")
    df = (
        session.read
               .option("header", "true")
               .csv(item["path"])
    )
    df.write.save_as_table(item["table"], mode="overwrite")
    print("  OK")

Ingesting → bronze.crm_cust_info


  OK
Ingesting → bronze.crm_prd_info


  OK
Ingesting → bronze.crm_sales_details


  OK
Ingesting → bronze.erp_cust_az12


  OK
Ingesting → bronze.erp_loc_a101


  OK
Ingesting → bronze.erp_px_cat_g1v2


  OK


## Sanity Check

In [4]:
session.table("bronze.crm_cust_info").limit(5).show()

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|
+------+----------+-------------+------------+------------------+--------+---------------+

